# Dataset Augmentation with Synthetic Waste Records

## Objective
This notebook expands the original food waste dataset by generating realistic synthetic records. The goal is to create a larger dataset that preserves the statistical properties of the real data while extending it to a longer time period (January 2025 to December 2028). The synthetic data includes:
- Realistic timestamps at meal times
- Foot traffic patterns that vary by day of week and special events
- Event indicators to mark days with higher waste (e.g., campus events)
- Consistent 30‑minute temporal granularity

The final output combines original and synthetic records into a single CSV file, ready for time‑series analysis and forecasting.

The time range is chosen to achieve approximately 10,000 non‑zero 30‑minute windows, which is suitable for robust forecasting models.

## 1. Setup and Data Loading
We import required libraries and load the original cleaned dataset. A random seed is set to ensure reproducible results.

In [103]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta, datetime

In [104]:
np.random.seed(42)

In [105]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change working directory to project folder
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


In [106]:
df_original = pd.read_csv('data/food_waste_cleaned.csv')
print(f"Original data shape: {df_original.shape}")

Original data shape: (2600, 23)


## 2. Add Realistic Times to Original Records
The original dataset only contains dates. We assign an hour and minute based on the recorded meal type to create timestamps that reflect typical meal times. This step makes the original data compatible with time‑series analysis.

In [107]:
# Define meal time windows (start hour, end hour)
meal_time_windows = {
    "Breakfast": (6, 9),
    "Lunch": (11, 14),
    "Dinner": (17, 20)
}

def add_realistic_times(df):
    """
    Convert date column to datetime and add a realistic time within the meal window.
    """
    df['Date'] = pd.to_datetime(df['Date'])

    def assign_time(row):
        start, end = meal_time_windows[row['Meal']]
        hour = np.random.randint(start, end)
        minute = np.random.randint(0, 60)
        return row['Date'] + timedelta(hours=hour, minutes=minute)

    df['Date'] = df.apply(assign_time, axis=1)
    return df

df_original = add_realistic_times(df_original)
print("First few timestamps after adding times:")
print(df_original['Date'].head())

First few timestamps after adding times:
0   2025-06-11 08:51:00
1   2025-06-11 11:14:00
2   2025-06-11 19:07:00
3   2025-06-11 11:20:00
4   2025-06-11 19:57:00
Name: Date, dtype: datetime64[ns]


## 3. Analyse Original Data
We examine the distributions of categorical and numerical variables from the original data. These statistics will guide the synthetic data generation to ensure it mimics real patterns.

In [108]:
# Categorical distributions
meal_dist = df_original['Meal'].value_counts(normalize=True)
section_dist = df_original['Canteen_Section'].value_counts(normalize=True)
category_dist = df_original['Food_Category'].value_counts(normalize=True)

# Compute conditional distributions (e.g., Food_Category given Meal)
category_given_meal = df_original.groupby('Meal')['Food_Category'].value_counts(normalize=True).unstack(fill_value=0)

print("Meal distribution:\n", meal_dist)
print("\nCanteen Section distribution:\n", section_dist)
print("\nFood Category distribution:\n", category_dist)
print("\nFood Category given Meal:\n", category_given_meal)

Meal distribution:
 Meal
Breakfast    0.334231
Lunch        0.333462
Dinner       0.332308
Name: proportion, dtype: float64

Canteen Section distribution:
 Canteen_Section
B    0.251538
A    0.250769
D    0.249231
C    0.248462
Name: proportion, dtype: float64

Food Category distribution:
 Food_Category
Rice          0.253462
Meat          0.251154
Vegetables    0.250385
Soup          0.245000
Name: proportion, dtype: float64

Food Category given Meal:
 Food_Category      Meat      Rice      Soup  Vegetables
Meal                                                   
Breakfast      0.245109  0.249712  0.255466    0.249712
Dinner         0.255787  0.261574  0.237269    0.245370
Lunch          0.252595  0.249135  0.242215    0.256055


In [109]:
# Unit price per food category (fixed)
price_map = {'Meat': 8.0, 'Vegetables': 3.0, 'Soup': 1.5, 'Rice': 2.0}

# Waste weight distribution per category (log-normal)
df_original['Log_Waste'] = np.log(df_original['Waste_Weight_kg'])
category_params = {}
for cat in df_original['Food_Category'].unique():
    log_vals = df_original[df_original['Food_Category'] == cat]['Log_Waste']
    category_params[cat] = {'mean_log': log_vals.mean(), 'std_log': log_vals.std()}
    print(f"{cat}: log mean = {category_params[cat]['mean_log']:.3f}, log std = {category_params[cat]['std_log']:.3f}")

Vegetables: log mean = 0.721, log std = 0.802
Soup: log mean = 0.756, log std = 0.787
Meat: log mean = 0.643, log std = 0.851
Rice: log mean = 0.730, log std = 0.793


In [110]:
# Compute baseline event counts per meal and day-of-week
# We group the original data by date and meal to count how many events occur per meal per day
# Then average over days of week to get a baseline intensity
df_original['DayOfWeek'] = df_original['Date'].dt.dayofweek
events_per_meal_day = df_original.groupby(['Date', 'Meal']).size().reset_index(name='count')
events_per_meal_day['DayOfWeek'] = events_per_meal_day['Date'].dt.dayofweek
baseline_events = events_per_meal_day.groupby(['Meal', 'DayOfWeek'])['count'].mean().round().astype(int)
print("Baseline events per meal per day (by day of week):\n", baseline_events)

Baseline events per meal per day (by day of week):
 Meal       DayOfWeek
Breakfast  0            1
           1            1
           2            1
           3            1
           4            1
           5            1
           6            1
Dinner     0            1
           1            1
           2            1
           3            1
           4            1
           5            1
           6            1
Lunch      0            1
           1            1
           2            1
           3            1
           4            1
           5            1
           6            1
Name: count, dtype: int64


## 4. Define Global Thresholds
We calculate outlier and high‑waste thresholds from the original data. These thresholds will be applied to both original and synthetic records to ensure consistent flagging.

In [111]:
def compute_thresholds(df):
    """
    Compute IQR-based outlier bounds and 90th percentile thresholds for waste and cost.
    """
    q1_waste = df['Waste_Weight_kg'].quantile(0.25)
    q3_waste = df['Waste_Weight_kg'].quantile(0.75)
    iqr_waste = q3_waste - q1_waste
    waste_lower = q1_waste - 1.5 * iqr_waste
    waste_upper = q3_waste + 1.5 * iqr_waste

    q1_cost = df['Cost_Loss'].quantile(0.25)
    q3_cost = df['Cost_Loss'].quantile(0.75)
    iqr_cost = q3_cost - q1_cost
    cost_lower = q1_cost - 1.5 * iqr_cost
    cost_upper = q3_cost + 1.5 * iqr_cost

    high_waste_thresh = df['Waste_Weight_kg'].quantile(0.90)
    high_cost_thresh = df['Cost_Loss'].quantile(0.90)

    return {
        'waste_lower': waste_lower,
        'waste_upper': waste_upper,
        'cost_lower': cost_lower,
        'cost_upper': cost_upper,
        'high_waste': high_waste_thresh,
        'high_cost': high_cost_thresh
    }

thresholds = compute_thresholds(df_original)
print(f"Waste outlier bounds: [{thresholds['waste_lower']:.2f}, {thresholds['waste_upper']:.2f}]")
print(f"Cost outlier bounds: [{thresholds['cost_lower']:.2f}, {thresholds['cost_upper']:.2f}]")
print(f"High waste threshold (90th percentile): {thresholds['high_waste']:.2f}")
print(f"High cost threshold (90th percentile): {thresholds['high_cost']:.2f}")

Waste outlier bounds: [-2.23, 7.42]
Cost outlier bounds: [-7.69, 22.13]
High waste threshold (90th percentile): 4.52
High cost threshold (90th percentile): 23.77


## 5. Define Helper Functions for Synthetic Generation
We break the generation process into small, reusable functions. This improves readability and makes it easy to adjust individual parts.

In [112]:
def get_foot_traffic_multiplier(date, event_days):
    """
    Return a multiplier that scales the expected number of waste events.
    Higher values represent more foot traffic.
    Factors:
    - Weekends have 0.7 times the traffic of weekdays
    - Event days have 1.8 times the traffic
    """
    weekday = date.weekday()
    if weekday >= 5:  # Saturday or Sunday
        multiplier = 0.7
    else:
        multiplier = 1.0

    if date in event_days:
        multiplier *= 1.8
    return multiplier

In [113]:
def generate_event_count(meal, day_of_week, foot_traffic_multiplier, baseline_events):
    """
    Sample the number of waste events for a given meal on a specific day.
    Uses Poisson distribution with lambda = baseline * foot_traffic_multiplier.
    """
    base = baseline_events.loc[(meal, day_of_week)]
    lam = base * foot_traffic_multiplier
    return np.random.poisson(lam)

In [114]:
def generate_event_timestamp(base_date, meal):
    """
    Create a random timestamp within the meal window.
    The minute is rounded to the nearest 30-minute bucket (0 or 30).
    """
    start_hour, end_hour = meal_time_windows[meal]
    hour = int(np.random.randint(start_hour, end_hour))
    # Minute bucket: 0 or 30
    minute_bucket = int(np.random.choice([0, 30]))
    # Random offset within the 30-minute interval (0 to 29 minutes)
    minute_offset = int(np.random.randint(0, 30))
    return base_date + timedelta(hours=hour, minutes=minute_bucket + minute_offset)

In [115]:
def generate_waste_weight(category, meal, section, foot_traffic_multiplier, event_multiplier=1.0):
    """
    Generate a waste weight for a single event.
    Steps:
    1. Draw from log-normal distribution of the category
    2. Multiply by meal-specific factor (breakfast:0.8, lunch:1.3, dinner:1.1)
    3. Multiply by section-specific factor (A:1.0, B:1.1, C:0.9, D:1.2)
    4. Multiply by foot traffic multiplier (higher traffic -> more waste)
    5. Multiply by event multiplier (for special days)
    """
    meal_multiplier = {'Breakfast': 0.8, 'Lunch': 1.3, 'Dinner': 1.1}
    section_multiplier = {'A': 1.0, 'B': 1.1, 'C': 0.9, 'D': 1.2}

    params = category_params[category]
    log_waste = np.random.normal(params['mean_log'], params['std_log'])
    base_waste = np.exp(log_waste)

    waste = base_waste * meal_multiplier[meal] * section_multiplier[section]
    waste *= foot_traffic_multiplier
    waste *= event_multiplier
    return max(waste, 0.01)  # ensure positive

In [116]:
def generate_derived_features(timestamp, waste_weight, unit_price):
    """
    Compute all date-time derived columns and flag columns based on thresholds.
    """
    cost_loss = waste_weight * unit_price
    waste_per_price = waste_weight / unit_price if unit_price > 0 else 0
    log_waste = np.log(waste_weight)

    # Date-time components
    year = timestamp.year
    month = timestamp.month
    day = timestamp.day
    hour = timestamp.hour
    minute = timestamp.minute
    weekday_num = timestamp.weekday()
    weekday_name = timestamp.strftime('%A')
    week = timestamp.isocalendar()[1]
    day_of_year = timestamp.timetuple().tm_yday
    quarter = (month - 1) // 3 + 1
    is_weekend = 1 if weekday_num >= 5 else 0
    month_name = timestamp.strftime('%B')
    weekday_type = 'Weekend' if is_weekend else 'Weekday'

    return {
        'Year': year, 'Month': month, 'Day': day, 'Hour': hour, 'Minute': minute,
        'Weekday': weekday_name, 'Week': week, 'DayOfYear': day_of_year, 'Quarter': quarter,
        'IsWeekend': is_weekend, 'Month_Name': month_name, 'Weekday_Type': weekday_type,
        'Cost_Loss': round(cost_loss, 2),
        'Waste_per_Price': round(waste_per_price, 6),
        'Log_Waste': round(log_waste, 6)
    }

In [117]:
def apply_flag_thresholds(df, thresholds):
    """
    Add boolean flag columns to the dataframe based on global thresholds.
    """
    df['Is_Waste_Outlier'] = ((df['Waste_Weight_kg'] < thresholds['waste_lower']) |
                              (df['Waste_Weight_kg'] > thresholds['waste_upper'])).astype(int)
    df['Is_Cost_Outlier'] = ((df['Cost_Loss'] < thresholds['cost_lower']) |
                             (df['Cost_Loss'] > thresholds['cost_upper'])).astype(int)
    df['Is_High_Waste'] = (df['Waste_Weight_kg'] > thresholds['high_waste']).astype(int)
    df['Is_High_Cost'] = (df['Cost_Loss'] > thresholds['high_cost']).astype(int)
    return df

## 6. Define Event Days
We randomly select some days within the target period to be event days. Event days simulate special occasions (e.g., campus festivals, sports events) that cause higher waste generation. We also create a dictionary mapping each event day to a multiplier (e.g., 1.5x) and possibly a section multiplier for targeted sections (like section D may have more waste on event days).

In [118]:
def create_event_days(start_date, end_date, n_events=10, seed=42):
    """
    Randomly select n_events unique days between start_date and end_date.
    Returns a set of dates.
    """
    np.random.seed(seed)
    date_range = pd.date_range(start_date, end_date)
    event_dates = np.random.choice(date_range, size=n_events, replace=False)
    return set(event_dates)

target_start = datetime(2015, 1, 1)
target_end = datetime(2024, 12, 31)
# Number of event days scaled proportionally to the original (15 events over ~7 months ≈ 2.14 per month, so ~103 for 48 months)
event_days = create_event_days(target_start, target_end, n_events=100, seed=42)

print(f"Selected {len(event_days)} event days:")
print(sorted(event_days)[:5], "...")

Selected 100 event days:
[np.datetime64('2015-01-18T00:00:00.000000000'), np.datetime64('2015-01-27T00:00:00.000000000'), np.datetime64('2015-02-02T00:00:00.000000000'), np.datetime64('2015-04-04T00:00:00.000000000'), np.datetime64('2015-04-20T00:00:00.000000000')] ...


## 7. Generate Synthetic Records
We loop over each day in the target period and for each meal generate a batch of events based on foot traffic and event multipliers. Each event is assigned a timestamp, categorical values, waste weight, and derived features. The synthetic dataset is built incrementally.

In [119]:
synthetic_rows = []
date_range = pd.date_range(target_start, target_end, freq='D')

for date in date_range:
    # Foot traffic multiplier for this day
    traffic_mult = get_foot_traffic_multiplier(date, event_days)

    for meal in meal_dist.index:
        # Determine how many events to generate for this meal on this day
        n_events = generate_event_count(meal, date.weekday(), traffic_mult, baseline_events)

        if n_events == 0:
            continue

        # For each event, sample categoricals and generate waste
        for _ in range(n_events):
            # Sample section and food category
            section = np.random.choice(section_dist.index, p=section_dist.values)
            # Use conditional distribution for food category given meal
            cat_probs = category_given_meal.loc[meal]
            category = np.random.choice(cat_probs.index, p=cat_probs.values)

            # Generate timestamp
            timestamp = generate_event_timestamp(date, meal)

            # Event multiplier (for event days, we can increase waste per event)
            event_mult = 1.5 if date in event_days else 1.0
            # Optionally, for event days we could increase section D more
            if date in event_days and section == 'D':
                event_mult *= 1.3

            waste_weight = generate_waste_weight(category, meal, section, traffic_mult, event_mult)
            unit_price = price_map[category]

            # Create base row
            row = {
                'Date': timestamp,
                'Meal': meal,
                'Canteen_Section': section,
                'Food_Category': category,
                'Waste_Weight_kg': round(waste_weight, 2),
                'Unit_Price_per_kg': unit_price
            }
            # Add derived features
            derived = generate_derived_features(timestamp, waste_weight, unit_price)
            row.update(derived)

            synthetic_rows.append(row)

df_synthetic = pd.DataFrame(synthetic_rows)
print(f"Synthetic data shape: {df_synthetic.shape}")

Synthetic data shape: (9987, 21)


## 8. Add Flag Columns to Synthetic Data
We apply the same outlier and high‑waste thresholds to the synthetic records to maintain consistency with the original data.

In [120]:
df_synthetic = apply_flag_thresholds(df_synthetic, thresholds)
print("Flag columns added.")

Flag columns added.


## 9. Combine Original and Synthetic Data
We merge the original dataset (with added realistic times) and the synthetic dataset. Both now have the same structure and flags. We then sort by date to ensure chronological order.

In [121]:
# Ensure original has all required columns (it should, but we add any missing ones)
df_original_clean = df_original.copy()
required_cols = df_synthetic.columns.tolist()
for col in required_cols:
    if col not in df_original_clean.columns:
        # For columns that are missing in original, we can compute them
        if col == 'Hour':
            df_original_clean['Hour'] = df_original_clean['Date'].dt.hour
        elif col == 'Minute':
            df_original_clean['Minute'] = df_original_clean['Date'].dt.minute
        elif col == 'DayOfYear':
            df_original_clean['DayOfYear'] = df_original_clean['Date'].dt.dayofyear
        elif col == 'Weekday':
            df_original_clean['Weekday'] = df_original_clean['Date'].dt.day_name()
        elif col == 'Week':
            df_original_clean['Week'] = df_original_clean['Date'].dt.isocalendar().week
        elif col == 'Quarter':
            df_original_clean['Quarter'] = df_original_clean['Date'].dt.quarter
        elif col == 'IsWeekend':
            df_original_clean['IsWeekend'] = (df_original_clean['Date'].dt.dayofweek >= 5).astype(int)
        elif col == 'Month_Name':
            df_original_clean['Month_Name'] = df_original_clean['Date'].dt.strftime('%B')
        elif col == 'Weekday_Type':
            df_original_clean['Weekday_Type'] = df_original_clean['IsWeekend'].map({0:'Weekday',1:'Weekend'})
        elif col == 'Waste_per_Price':
            df_original_clean['Waste_per_Price'] = df_original_clean['Waste_Weight_kg'] / df_original_clean['Unit_Price_per_kg']
        elif col == 'Log_Waste':
            df_original_clean['Log_Waste'] = np.log(df_original_clean['Waste_Weight_kg'])
        else:
            # If still missing, set to NaN
            df_original_clean[col] = np.nan

# Apply flag thresholds to original as well (they already have them, but recalc to be safe)
df_original_clean = apply_flag_thresholds(df_original_clean, thresholds)

# Combine
df_combined = pd.concat([df_original_clean, df_synthetic], ignore_index=True)
df_combined = df_combined.sort_values('Date').reset_index(drop=True)
print(f"Combined data shape: {df_combined.shape}")

Combined data shape: (12587, 26)


## 10. Save Augmented Dataset
Finally, we save the combined dataset to a CSV file. This file contains both original and synthetic records with full timestamps, ready for further analysis and modeling.

In [122]:
output_filename = 'data/food_waste_augmented.csv'
df_combined.to_csv(output_filename, index=False)
print(f"File saved as {output_filename}")

File saved as data/food_waste_augmented.csv


## 11. Quick Verification of Temporal Coverage
We check that the combined dataset covers the full target period and that 30‑minute intervals have a reasonable mix of zero and non‑zero values. This ensures the data is suitable for time‑series forecasting.

In [123]:
# Ensure datetime index
df_combined = df_combined.sort_values("Date")
agg = (
    df_combined.set_index("Date")
    .resample("30min")["Waste_Weight_kg"]
    .sum()
)

total_windows = len(agg)
zero_windows = (agg == 0).sum()
nonzero_windows = (agg > 0).sum()

print(f"Total 30‑min windows: {total_windows}")
print(f"Zero windows: {zero_windows} ({zero_windows/total_windows*100:.1f}%)")
print(f"Non‑zero windows: {nonzero_windows} ({nonzero_windows/total_windows*100:.1f}%)")

# Also check date range
print(f"Date range in combined data: {df_combined['Date'].min()} to {df_combined['Date'].max()}")

Total 30‑min windows: 185968
Zero windows: 175715 (94.5%)
Non‑zero windows: 10253 (5.5%)
Date range in combined data: 2015-01-01 12:00:00 to 2025-08-10 19:59:00
